# 04 RQ1 Bottleneck Forecast

Synopsis RQ1: predict forward bottleneck / congestion stress with spatial-temporal style features.

Models: naive lag baseline, Ridge, Random Forest. Validation: walk-forward time split.


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from project_paths import load_env, ensure_output_dirs
ROOT = load_env()
PATHS = ensure_output_dirs(ROOT)
print("ROOT", ROOT)
print({k: str(v) for k, v in PATHS.items()})

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error, accuracy_score, f1_score, roc_auc_score, confusion_matrix, RocCurveDisplay
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
sns.set_theme(style="whitegrid", context="talk")
df = pd.read_csv(PATHS["processed"] / "daily_panel_clean.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
print(df.shape)


In [ ]:
# Feature set locked to synopsis physical + market lags
feature_cols = [c for c in [
    "chokepoint_vessel_count","port_congestion_hours","vessel_draught_anomaly",
    "commodity_roll_yield","wti_spot","natgas_hh"
] if c in df.columns and df[c].notna().sum() >= 50]
if not feature_cols:
    raise RuntimeError("No usable RQ1 features after coverage filter")
work = df.dropna(subset=feature_cols + ["bottleneck_index_5d"]).copy()
work["lag1_target"] = work["bottleneck_index_5d"].shift(1)
work = work.dropna().reset_index(drop=True)

# Walk-forward: train 70%, test last 30% chronologically
split = int(len(work) * 0.7)
train, test = work.iloc[:split], work.iloc[split:]
X_train, y_train = train[feature_cols], train["bottleneck_index_5d"]
X_test, y_test = test[feature_cols], test["bottleneck_index_5d"]

# Baseline: lag1
pred_base = test["lag1_target"]
rmse_base = mean_squared_error(y_test, pred_base) ** 0.5
mae_base = mean_absolute_error(y_test, pred_base)

scaler = StandardScaler()
Xtr = scaler.fit_transform(X_train)
Xte = scaler.transform(X_test)
ridge = Ridge(alpha=1.0).fit(Xtr, y_train)
pred_ridge = ridge.predict(Xte)
rf = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=3).fit(X_train, y_train)
pred_rf = rf.predict(X_test)

rows = []
for name, pred in [("lag_baseline", pred_base), ("ridge", pred_ridge), ("random_forest", pred_rf)]:
    rows.append({
        "Model": name,
        "Feature Set": ",".join(feature_cols),
        "Validation Approach": "chronological 70/30 walk-forward",
        "RMSE": mean_squared_error(y_test, pred) ** 0.5,
        "MAE": mean_absolute_error(y_test, pred),
        "Key Takeaway": "",
    })
metrics = pd.DataFrame(rows)
best = metrics.sort_values("RMSE").iloc[0]["Model"]
metrics.loc[metrics["Model"]==best, "Key Takeaway"] = "Best RMSE on holdout window"
metrics.to_csv(PATHS["outputs"] / "metrics_rq1.csv", index=False)
metrics.to_csv(PATHS["tables"] / "model_performance_rq1.csv", index=False)
metrics


### Figure 4. RQ1 forecast vs actual (best model on holdout)


In [ ]:
pred_best = {"lag_baseline": pred_base, "ridge": pred_ridge, "random_forest": pred_rf}[best]
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(test["date"], y_test.values, label="Actual", color="#1b263b")
ax.plot(test["date"], pred_best, label=f"Predicted ({best})", color="#e76f51", alpha=0.85)
ax.set_title("Figure 4. RQ1 bottleneck forecast vs actual (holdout)")
ax.set_xlabel("Date")
ax.set_ylabel("Bottleneck index")
ax.legend()
fig.tight_layout()
fig.savefig(PATHS["figures"] / "fig04_rq1_forecast.png", dpi=200)
plt.show()
print(metrics)
